# Create Triggered Task — Interactive Notebook

Companion notebook for [`01.create_triggered_task.robot`](./01.create_triggered_task.robot).
Covers **all 4 test cases + all 11 argument-passing patterns** from the Robot file.

## What this notebook covers

1. **Setup** — env, session, fetch `ORG_SNODE_ID` and `runtime_path_id`
2. **Helpers** — import-pipeline + create-task wrappers
3. **Prereqs** — bulk-import all 12 pipelines this notebook needs (11 numbered + 1 child)
4. **Cases 1–11** — the 11 argument-passing patterns from `Create Triggered Task For Pipeline`
5. **Case 12** — `Create Triggered Task For Original Pipeline Name` (no-suffix lookup)
6. **Verify** — list all created tasks
7. **Cleanup** — delete tasks + pipelines (optional)

## Prerequisites

- Start Jupyter via `make jupyter-start` (or `make start-services`).
- A **Groundplex must be running** (the Snaplex named in `GROUNDPLEX_NAME`).
- Required env vars (auto-loaded from `/app/.env`):
  - `URL`, `ORG_ADMIN_USER`, `ORG_ADMIN_PASSWORD`, `ORG_NAME`
  - `PIPELINES_LOCATION_PATH`
  - `GROUNDPLEX_NAME`

## SnapLogic API used here

| Operation | Method | Endpoint |
|---|---|---|
| Get ORG_SNODE_ID | `GET` | `/api/1/rest/asset/list/{ORG}` → `entries[0].parent_snode_id` |
| Get runtime_path_id | `GET` | `/api/1/rest/slserver/snaplex_cc_details_with_info?subscriber_id={ORG}` → match on `label == plex_name` |
| Import pipeline | `POST` | `/api/2/{ORG_SNODE_ID}/rest/project/import/pipe/slp/{ORG}/{path}?duplicate_check=false` |
| Create task | `POST` | `/api/1/rest/slsched/job?duplicate_check=False` (body from `triggered_task.json` template) |
| List | `GET` | `/api/1/rest/asset/list/{ORG}/{path}` (filter `asset_type="Task"` or `"Pipeline"`) |
| Delete | `DELETE` | `/api/1/rest/asset/{ORG}/{path}/{name}` |

## Section 1 — Setup

Load env vars, build a Basic-auth `requests.Session`, then fetch two values
the task-create endpoint requires:

- **`ORG_SNODE_ID`** — embedded in the pipeline-import URL path
- **`runtime_path_id`** — looked up by Groundplex label; goes inside the task payload

In [ ]:
import os
import json
import time
import secrets
import copy
from pathlib import Path
import requests
from dotenv import load_dotenv

load_dotenv("/app/.env", override=False)

SL_URL  = os.environ.get("URL", "https://elastic.snaplogic.com")
SL_USER = os.environ.get("ORG_ADMIN_USER")
SL_PASS = os.environ.get("ORG_ADMIN_PASSWORD")
ORG     = os.environ.get("ORG_NAME")
PIPELINES_LOCATION_PATH = os.environ.get("PIPELINES_LOCATION_PATH")
GROUNDPLEX_NAME         = os.environ.get("GROUNDPLEX_NAME")

missing = [k for k, v in {
    "ORG_ADMIN_USER": SL_USER, "ORG_ADMIN_PASSWORD": SL_PASS,
    "ORG_NAME": ORG, "PIPELINES_LOCATION_PATH": PIPELINES_LOCATION_PATH,
    "GROUNDPLEX_NAME": GROUNDPLEX_NAME,
}.items() if not v]
if missing:
    raise RuntimeError(f"Missing required env vars: {missing}")

session = requests.Session()
session.auth = (SL_USER, SL_PASS)
session.headers.update({"Content-Type": "application/json"})

# Fetch ORG_SNODE_ID (embedded in import URL)
r = session.get(f"{SL_URL}/api/1/rest/asset/list/{ORG}")
r.raise_for_status()
ORG_SNODE_ID = r.json()["response_map"]["entries"][0]["parent_snode_id"]

# Fetch runtime_path_id (goes inside task payload)
r = session.get(f"{SL_URL}/api/1/rest/slserver/snaplex_cc_details_with_info",
                params={"subscriber_id": ORG})
r.raise_for_status()
plex = next((p for p in r.json()["response_map"] if p.get("label") == GROUNDPLEX_NAME), None)
if not plex:
    raise RuntimeError(f"Groundplex '{GROUNDPLEX_NAME}' not found in org {ORG}")
RUNTIME_PATH_ID = plex["runtime_path_id"]

print(f"✅ SnapLogic session ready")
print(f"   URL:                     {SL_URL}")
print(f"   Org:                     {ORG}")
print(f"   PIPELINES_LOCATION_PATH: {PIPELINES_LOCATION_PATH}")
print(f"   GROUNDPLEX_NAME:         {GROUNDPLEX_NAME}")
print(f"   ORG_SNODE_ID:            {ORG_SNODE_ID}")
print(f"   RUNTIME_PATH_ID:         {RUNTIME_PATH_ID}")

## Section 2 — Helper functions

Two groups of helpers:

**Pipeline helpers** (same shape as in the [import_pipeline notebook](../03.pipelines/import_pipeline.ipynb)):
- `import_from_template(unique_id, ...)` — name becomes `<name>_<unique_id>`
- `import_with_original_name(...)` — name stays exactly as-is

**Task helpers** (this notebook's focus):
- `build_task_payload(...)` — clone the `triggered_task.json` template and patch in our values
- `create_triggered_task(task_name, pipeline_snode_id, ..., pipeline_params=None, notification=None, execution_timeout=None)` — POST to `/api/1/rest/slsched/job`
- `create_task_from_template(...)` — mirrors Robot's `Create Triggered Task From Template`: pipeline lookup via `<pipeline_name>_<unique_id>`
- `create_task_for_original_pipeline_name(...)` — mirrors the no-suffix variant
- `list_tasks(path)`, `delete_task(path, task_name)` — read / cleanup

In [ ]:
PIPELINE_PAYLOAD_PATH = Path("/app/src/pipelines")
TRIGGERED_TASK_TEMPLATE_PATH = Path(
    "/usr/local/lib/python3.12/site-packages/snaplogic_common_robot/test_data/triggered_task.json"
)

# Mirrors Robot's suite vars ${<name>_snode_id} / ${<full_task_name>_snodeid}
pipeline_snode_ids: dict[str, str] = {}
task_snode_ids: dict[str, str] = {}


def get_unique_id() -> str:
    return f"{int(time.time())}_{secrets.token_hex(3)}"


# --- Pipeline helpers --------------------------------------------------------
def _import_pipeline(slp_filename: str, pipeline_name: str, project_path: str, duplicate_check: bool = False) -> str:
    slp = json.loads((PIPELINE_PAYLOAD_PATH / slp_filename).read_text())
    slp["property_map"]["info"]["label"]["value"] = pipeline_name
    url = f"{SL_URL}/api/2/{ORG_SNODE_ID}/rest/project/import/pipe/slp/{ORG}/{project_path}"
    r = session.post(url, params={"duplicate_check": str(duplicate_check).lower()}, json={"pipe": slp})
    if not r.ok:
        print(f"❌ import {r.status_code}: {r.text[:400]}")
    r.raise_for_status()
    return r.json()["response_map"]


def import_from_template(unique_id: str, project_path: str, pipeline_name: str, slp_filename: str) -> str:
    """Mirrors Robot's `Import Pipelines From Template` — appends `_<unique_id>` to the name."""
    full_name = f"{pipeline_name}_{unique_id}"
    snode_id = _import_pipeline(slp_filename, full_name, project_path)
    pipeline_snode_ids[full_name] = snode_id
    return snode_id


def import_with_original_name(project_path: str, pipeline_name: str, slp_filename: str) -> str:
    """Mirrors Robot's `Import Pipeline With Original Name` — keeps name exactly as-is."""
    snode_id = _import_pipeline(slp_filename, pipeline_name, project_path)
    pipeline_snode_ids[pipeline_name] = snode_id
    return snode_id


# --- Task helpers ------------------------------------------------------------
def _print_task_summary(
    full_task_name: str,
    pipeline_key: str,
    pipeline_snode_id: str,
    plex_name: str,
    pipeline_params: dict | None,
    notification: dict | None,
    execution_timeout: int | None,
    task_snode_id: str,
) -> None:
    """Pretty-print everything that went into (and came out of) the task creation."""
    print("┌─ Created Task ─────────────────────────────────────────────────────────────")
    print(f"│ Task name      : {full_task_name}")
    print(f"│ Task snode_id  : {task_snode_id}")
    print(f"│ Plex           : {plex_name}")
    print(f"│ Pipeline       : {pipeline_key}")
    print(f"│ Pipeline snode : {pipeline_snode_id}")
    print("├─ Optional fields applied to the payload ──────────────────────────────────")
    print(f"│ pipeline_params    : {pipeline_params if pipeline_params is not None else '— not provided (omitted from payload)'}")
    print(f"│ notification       : {notification        if notification        is not None else '— not provided (omitted from payload)'}")
    print(f"│ execution_timeout  : {execution_timeout   if execution_timeout   is not None else '— not provided (omitted from payload)'}")
    print("└────────────────────────────────────────────────────────────────────────────")


def build_task_payload(
    task_name: str,
    pipeline_snode_id: str,
    runtime_path_id: str,
    project_path: str,
    pipeline_params: dict | None = None,
    notification: dict | None = None,
    execution_timeout: int | None = None,
) -> dict:
    """Build the task payload by cloning the `triggered_task.json` template and patching fields.

    Optional fields are only included when their argument is not None — matches the
    Robot keyword's `IF $X is not None` logic.
    """
    payload = copy.deepcopy(json.loads(TRIGGERED_TASK_TEMPLATE_PATH.read_text()))
    payload["job_name"] = task_name
    payload["name"] = task_name
    payload["path_id"] = f"{ORG}/{project_path}"
    payload["org_path"] = f"/{ORG}"
    p = payload["parameters"]
    p["runtime_path_id"] = runtime_path_id
    p["pipeline_snode_id"] = pipeline_snode_id
    if pipeline_params is not None:
        p.setdefault("pipeline_parameters", {}).update(pipeline_params)
    if execution_timeout is not None:
        p["execution_timeout"] = int(execution_timeout)
    if notification is not None:
        p["notification"] = notification
    return payload


def create_triggered_task(
    task_name: str,
    pipeline_snode_id: str,
    plex_name: str,
    project_path: str,
    pipeline_params: dict | None = None,
    notification: dict | None = None,
    execution_timeout: int | None = None,
) -> str:
    """Core API call — mirrors Robot's `Create Triggered Task` keyword."""
    r = session.get(f"{SL_URL}/api/1/rest/slserver/snaplex_cc_details_with_info",
                    params={"subscriber_id": ORG})
    r.raise_for_status()
    plex = next((p for p in r.json()["response_map"] if p.get("label") == plex_name), None)
    if not plex:
        raise RuntimeError(f"Plex not found: {plex_name}")
    rpi = plex["runtime_path_id"]

    payload = build_task_payload(
        task_name, pipeline_snode_id, rpi, project_path,
        pipeline_params=pipeline_params,
        notification=notification,
        execution_timeout=execution_timeout,
    )
    r = session.post(f"{SL_URL}/api/1/rest/slsched/job",
                     params={"duplicate_check": "False"}, json=payload)
    if not r.ok:
        print(f"❌ create_task {r.status_code}: {r.text[:500]}")
    r.raise_for_status()
    return r.json()["response_map"]["snode_id"]


def create_task_from_template(
    unique_id: str,
    project_path: str,
    pipeline_name: str,
    task_name: str,
    plex_name: str = None,
    pipeline_params: dict | None = None,
    notification: dict | None = None,
    execution_timeout: int | None = None,
) -> tuple[str, str]:
    """Mirrors Robot's `Create Triggered Task From Template`.

    - Pipeline lookup: `pipeline_snode_ids[f"{pipeline_name}_{unique_id}"]`
    - Task name: `f"{pipeline_name}_{task_name}_{unique_id}"`
    - `plex_name=None` falls back to the global `GROUNDPLEX_NAME` (matches Robot's default).
    """
    plex = plex_name or GROUNDPLEX_NAME
    full_task_name = f"{pipeline_name}_{task_name}_{unique_id}"
    pipeline_key = f"{pipeline_name}_{unique_id}"
    pipeline_snode_id = pipeline_snode_ids.get(pipeline_key)
    if not pipeline_snode_id:
        raise RuntimeError(f"No imported pipeline named {pipeline_key} — import it first")
    snode_id = create_triggered_task(
        full_task_name, pipeline_snode_id, plex, project_path,
        pipeline_params=pipeline_params,
        notification=notification,
        execution_timeout=execution_timeout,
    )
    task_snode_ids[full_task_name] = snode_id
    _print_task_summary(full_task_name, pipeline_key, pipeline_snode_id,
                        plex, pipeline_params, notification, execution_timeout, snode_id)
    return full_task_name, snode_id


def create_task_for_original_pipeline_name(
    unique_id: str,
    project_path: str,
    pipeline_name: str,
    task_name: str,
    plex_name: str = None,
    pipeline_params: dict | None = None,
    notification: dict | None = None,
    execution_timeout: int | None = None,
) -> tuple[str, str]:
    """Mirrors Robot's `Create Triggered Task For Original Pipeline Name`.

    Pipeline lookup uses the original name (NO unique_id suffix): `pipeline_snode_ids[pipeline_name]`.
    Task name still includes `_<unique_id>` to avoid collisions.
    """
    plex = plex_name or GROUNDPLEX_NAME
    full_task_name = f"{pipeline_name}_{task_name}_{unique_id}"
    pipeline_snode_id = pipeline_snode_ids.get(pipeline_name)
    if not pipeline_snode_id:
        raise RuntimeError(f"No imported pipeline named {pipeline_name} — import it first")
    snode_id = create_triggered_task(
        full_task_name, pipeline_snode_id, plex, project_path,
        pipeline_params=pipeline_params,
        notification=notification,
        execution_timeout=execution_timeout,
    )
    task_snode_ids[full_task_name] = snode_id
    _print_task_summary(full_task_name, f"{pipeline_name}  (ORIGINAL — no _unique_id suffix)",
                        pipeline_snode_id, plex, pipeline_params, notification, execution_timeout, snode_id)
    return full_task_name, snode_id


def list_assets(project_path: str, asset_type: str) -> list[dict]:
    r = session.get(f"{SL_URL}/api/1/rest/asset/list/{ORG}/{project_path}")
    r.raise_for_status()
    return [e for e in r.json()["response_map"]["entries"] if e.get("asset_type") == asset_type]


def delete_asset(project_path: str, name: str) -> bool:
    r = session.delete(f"{SL_URL}/api/1/rest/asset/{ORG}/{project_path}/{name}")
    return r.ok


# Generate the one unique_id for this notebook run
unique_id = get_unique_id()
print(f"✅ Helper functions defined")
print(f"   unique_id for this run: {unique_id}")

## Section 3 — Prereq: Import the 12 pipelines this notebook needs

The Robot test imports **11 numbered pipelines** (one per task case) plus the
**child pipeline with original name**. The task creation steps later look up
pipeline snode_ids by name — we import them up-front so all task cases find their pipeline.

Pipeline-name suffixes used:

| Used by case | Pipeline name |
|---|---|
| Case 1 | `prime_oracle_baseline_tests3_<unique_id>` |
| Case 2 | `prime_oracle_baseline_tests3_<unique_id>_2` |
| Case 3 | `prime_oracle_baseline_tests3_<unique_id>_3` |
| ... | ... |
| Case 11 | `prime_oracle_baseline_tests3_<unique_id>_11` |
| Case 12 | `prime_oracle_child_pipeline` (no suffix) |

In [ ]:
# 11 numbered pipelines — suffixes match the Robot test rows
for suffix in [""] + [f"_{i}" for i in range(2, 12)]:
    uid = f"{unique_id}{suffix}"
    snode_id = import_from_template(
        unique_id=uid,
        project_path=PIPELINES_LOCATION_PATH,
        pipeline_name="prime_oracle_baseline_tests3",
        slp_filename="prime_oracle_baseline_tests.slp",
    )
    print(f"   imported  prime_oracle_baseline_tests3_{uid}  →  {snode_id}")

# Child pipeline — original name, no suffix
snode_id = import_with_original_name(
    project_path=PIPELINES_LOCATION_PATH,
    pipeline_name="prime_oracle_child_pipeline",
    slp_filename="prime_oracle_child_pipeline.slp",
)
print(f"   imported  prime_oracle_child_pipeline  →  {snode_id}")

print(f"\n✅ Imported {len(pipeline_snode_ids)} pipelines for this notebook run")

## Shared test variables (mirrors the Robot `*** Variables ***` block)

These are the optional dicts/values the 11 cases below pass around.

In [ ]:
pipeline_name = "prime_oracle_baseline_tests3"   # base name (no unique_id)
task1         = "My_Task"                         # task name prefix

task_params_set     = {"oracle_acct": "../shared/oracle_acct"}
task_notifications  = {
    "recipients": "demo@example.com",
    "states":     ["Completed", "Failed"],
}
task_timeout = 300

print("Shared params ready:")
print(f"  pipeline_name      = {pipeline_name}")
print(f"  task1              = {task1}")
print(f"  task_params_set    = {task_params_set}")
print(f"  task_notifications = {task_notifications}")
print(f"  task_timeout       = {task_timeout}")

## Case 1 — BARE MINIMUM (4 required args only)

```robot
${unique_id}    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}
```

`plex_name` defaults to `GROUNDPLEX_NAME`; params/notification/timeout default to `None`.

In [ ]:
create_task_from_template(unique_id, PIPELINES_LOCATION_PATH, pipeline_name, task1)

## Case 2 — ALL POSITIONAL (full 8 args)

```robot
${unique_id}_2    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    ${GROUNDPLEX_NAME}    ${task_params_set}    ${task_notifications}    ${task_timeout}
```

In [ ]:
create_task_from_template(
    f"{unique_id}_2", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    GROUNDPLEX_NAME, task_params_set, task_notifications, task_timeout,
)

## Case 3 — 7 POSITIONAL (drop trailing `timeout`)

```robot
${unique_id}_3    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    ${GROUNDPLEX_NAME}    ${task_params_set}    ${task_notifications}
```

In [ ]:
create_task_from_template(
    f"{unique_id}_3", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    GROUNDPLEX_NAME, task_params_set, task_notifications,
)

## Case 4 — PLEX OVERRIDE ONLY (5 positional)

```robot
${unique_id}_4    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    ${GROUNDPLEX_NAME}
```

Same as Case 1 but explicit plex name (still our default plex).

In [ ]:
create_task_from_template(
    f"{unique_id}_4", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    GROUNDPLEX_NAME,
)

## Case 5 — PIPELINE PARAMS ONLY (named, skip plex)

```robot
${unique_id}_5    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    pipeline_params=${task_params_set}
```

In [ ]:
create_task_from_template(
    f"{unique_id}_5", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    pipeline_params=task_params_set,
)

## Case 6 — NOTIFICATIONS ONLY (named, skip plex AND params)

```robot
${unique_id}_6    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    notification=${task_notifications}
```

In [ ]:
create_task_from_template(
    f"{unique_id}_6", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    notification=task_notifications,
)

## Case 7 — TIMEOUT ONLY (named, skip plex / params / notification)

```robot
${unique_id}_7    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    execution_timeout=${task_timeout}
```

`plex_name` falls back to `GROUNDPLEX_NAME` — same default as Robot's `${groundplex_name}`.

In [ ]:
create_task_from_template(
    f"{unique_id}_7", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    execution_timeout=task_timeout,
)

## Case 8 — PLEX (positional) + NOTIFICATION (named), skip params

```robot
${unique_id}_8    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    ${GROUNDPLEX_NAME}    notification=${task_notifications}
```

In [ ]:
create_task_from_template(
    f"{unique_id}_8", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    GROUNDPLEX_NAME,
    notification=task_notifications,
)

## Case 9 — Multiple NAMED-ONLY optional args (skip in any order)

```robot
${unique_id}_9    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    notification=${task_notifications}    execution_timeout=${task_timeout}
```

In [ ]:
create_task_from_template(
    f"{unique_id}_9", PIPELINES_LOCATION_PATH, pipeline_name, task1,
    notification=task_notifications, execution_timeout=task_timeout,
)

## Case 10 — ALL NAMED (every argument by `name=value`)

```robot
unique_id=${unique_id}_10    project_path=${PIPELINES_LOCATION_PATH}    pipeline_name=${pipeline_name}    task_name=${task1}    plex_name=${GROUNDPLEX_NAME}    pipeline_params=${task_params_set}    notification=${task_notifications}    execution_timeout=${task_timeout}
```

Even required args can be named. Verbose but explicit.

In [ ]:
create_task_from_template(
    unique_id=f"{unique_id}_10",
    project_path=PIPELINES_LOCATION_PATH,
    pipeline_name=pipeline_name,
    task_name=task1,
    plex_name=GROUNDPLEX_NAME,
    pipeline_params=task_params_set,
    notification=task_notifications,
    execution_timeout=task_timeout,
)

## Case 11 — ALL NAMED in SCRAMBLED order

```robot
plex_name=${GROUNDPLEX_NAME}    task_name=${task1}    unique_id=${unique_id}    pipeline_name=${pipeline_name}    project_path=${PIPELINES_LOCATION_PATH}    notification=${task_notifications}
```

Demonstrates that with named args, order is irrelevant.

**Robot quirk**: this case uses `${unique_id}` (no suffix) — but Case 1 already created a task with that exact name, so re-running would conflict. We use `_11` here to keep the notebook re-runnable.

In [ ]:
create_task_from_template(
    plex_name=GROUNDPLEX_NAME,
    task_name=task1,
    unique_id=f"{unique_id}_11",
    pipeline_name=pipeline_name,
    project_path=PIPELINES_LOCATION_PATH,
    notification=task_notifications,
)

## Case 12 — Task for an ORIGINAL-NAMED pipeline (no suffix)

Mirrors the second test case `Create Triggered Task For Pipelines WithOut UniqueID Appended`:

```robot
${unique_id}    ${PIPELINES_LOCATION_PATH}    ${pipeline_name}    ${task1}    ${GROUNDPLEX_NAME}    ${task_params_set}    ${task_notifications}
```

Key difference: pipeline lookup uses `prime_oracle_child_pipeline` (no `_<unique_id>` suffix).
The task name still includes `_<unique_id>` to avoid collisions across runs.

In [ ]:
create_task_for_original_pipeline_name(
    unique_id=unique_id,
    project_path=PIPELINES_LOCATION_PATH,
    pipeline_name="prime_oracle_child_pipeline",
    task_name=task1,
    plex_name=GROUNDPLEX_NAME,
    pipeline_params=task_params_set,
    notification=task_notifications,
)

## Verify — list created tasks

Lists all Task assets at the project path and highlights the 12 we created.

In [ ]:
all_tasks = list_assets(PIPELINES_LOCATION_PATH, "Task")
my_task_names = set(task_snode_ids.keys())
print(f"📋 Task assets in {ORG}/{PIPELINES_LOCATION_PATH}: ({len(all_tasks)} total)")
for entry in sorted(all_tasks, key=lambda e: e["name"]):
    if entry["name"] in my_task_names:
        aid = (entry.get("asset_id") or "")[:16]
        print(f"   ⭐ {entry['name']:60s}  ({aid}...)")

found = {e["name"] for e in all_tasks}
missing = my_task_names - found
if missing:
    print(f"\n⚠️  Created but not found in list: {missing}")
else:
    print(f"\n✅ All {len(my_task_names)} tasks present in SnapLogic")

print(f"\n📌 task_snode_ids (mirrors Robot's ${{<full_task_name>_snodeid}} suite vars):")
for name, snode_id in task_snode_ids.items():
    print(f"   {name:60s} → {snode_id}")

## Cleanup (optional)

Delete all tasks AND pipelines created by this notebook run, leaving the project space clean.
Skip this if you want to inspect them in the SnapLogic Manager UI, or if you plan to
execute the tasks in the next tutorial folder (`02.execute_trigger_task.robot`).

In [ ]:
# Delete tasks first (they reference pipelines)
for name in list(task_snode_ids.keys()):
    if delete_asset(PIPELINES_LOCATION_PATH, name):
        print(f"🗑️  deleted task    {name}")
        del task_snode_ids[name]

# Then delete pipelines
for name in list(pipeline_snode_ids.keys()):
    if delete_asset(PIPELINES_LOCATION_PATH, name):
        print(f"🗑️  deleted pipeline {name}")
        del pipeline_snode_ids[name]

print(f"\n✅ Cleanup done — remaining tracked: {len(task_snode_ids)} tasks, {len(pipeline_snode_ids)} pipelines")

## What's next

- **Execute the tasks** — see [`02.execute_trigger_task.robot`](../02.execute_trigger_task.robot) for the run-task flow (`POST /api/1/rest/slsched/feed/...`). A notebook for that would mirror these patterns but POST to the run endpoint.
- **Tweak any case** — try a different `plex_name`, set a small `execution_timeout` to see a timeout error, or change `pipeline_params`.
- **Run the official Robot test instead** for production validation:
  ```bash
  make robot-run-tests-no-gp TAGS="create_triggeredtask_sample"
  ```

## Caveats vs the Robot keywords

This notebook is a simplified educational version. The Robot keywords additionally:

- Set **suite-level variables** like `${<full_task_name>_payload}` and `${<full_task_name>_snodeid}` (we use `task_snode_ids` dict instead)
- Use `Log Pretty JSON` for nicer task-payload logging
- Set `enabled=true` and other defaults from the loaded template — we keep the template's defaults, so behavior matches

For production tests, **use the Robot keywords**. This notebook is for learning what's underneath.

## Related

- [`01.create_triggered_task.robot`](./01.create_triggered_task.robot) — runnable Robot test (4 test cases, 12 rows total)
- [`../03.pipelines/import_pipeline.ipynb`](../03.pipelines/import_pipeline.ipynb) — sibling notebook for pipeline imports
- [`../01.accounts/create_account.ipynb`](../01.accounts/create_account.ipynb) — sibling notebook for accounts
- [`../02.upload_files/upload_files.ipynb`](../02.upload_files/upload_files.ipynb) — sibling notebook for SLFS uploads